# rl_torch

> Phase 12, the PyTorch side of `clue_rl`: log-probabilities of reply tokens (left padding, so only the reply positions' logits are computed) and checkpoints (adapters, optimizer state, step, configuration hash) with refusal of another configuration's checkpoint.

In [ ]:
#| default_exp rl_torch

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
from pathlib import Path

from explRL.local_policy import encodePair, promptText


def replyLogps(model, tok, samples, maxLen, device):
    "Per sample, the log-probabilities of its reply tokens (left padding, so the replies end the batch together)."
    import torch
    seqs = [encodePair(tok, promptText(tok, s['system'], s['blocks']), s['reply'], maxLen, s['eos']) for s in samples]
    L, K = max(len(x) for x, _ in seqs), max(len(x) - st for x, st in seqs)
    ids = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long)
    mask = torch.zeros_like(ids)
    for n, (x, _) in enumerate(seqs):
        ids[n, L - len(x):], mask[n, L - len(x):] = torch.tensor(x), 1
    ids, mask = ids.to(device), mask.to(device)
    pos = (mask.cumsum(-1) - 1).clamp(min=0)
    logits = model(input_ids=ids, attention_mask=mask, position_ids=pos, logits_to_keep=K + 1).logits[:, :-1].float()
    lp = torch.log_softmax(logits, -1).gather(-1, ids[:, L - K:, None])[..., 0]
    return [lp[n, K - (len(x) - st):] for n, (x, st) in enumerate(seqs)]


def _ckpts(out):
    return sorted(Path(out).glob('ckpt-*'))


def saveCheckpoint(model, opt, step, out, config):
    "out/ckpt-<step>: the adapters, the optimizer state, the step and the run's configuration hash."
    import torch
    d = Path(out) / f'ckpt-{step:04d}'
    model.save_pretrained(d)
    torch.save(opt.state_dict(), d / 'optimizer.pt')
    (d / 'state.json').write_text(json.dumps(dict(step=step, config=config)) + '\n')
    return d


def loadCheckpoint(model, opt, out, config, names):
    "Resume from the newest checkpoint: -> its step (0 without one); refuses one written by another configuration."
    import torch
    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file
    if not _ckpts(out):
        return 0
    d = _ckpts(out)[-1]
    st = json.loads((d / 'state.json').read_text())
    if st['config'] != config:
        raise ValueError(f'{d} was written by another configuration ({st["config"]} != {config})')
    for name in names:
        set_peft_model_state_dict(model, load_file(d / ('' if name == 'default' else name) / 'adapter_model.safetensors'),
                                  adapter_name=name)
    opt.load_state_dict(torch.load(d / 'optimizer.pt'))
    return st['step']

## Tests

In [ ]:
import inspect
from explRL.rl_torch import *
# the GPU-side pieces need the [train] extra; their behaviour is checked by clue_rl's train cell and the Colab pilot
assert list(inspect.signature(loadCheckpoint).parameters) == ['model', 'opt', 'out', 'config', 'names']
assert 'logits_to_keep' in inspect.getsource(replyLogps)       # only the reply positions' logits are computed

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()